# Capítulo 10 — Monitoramento em produção

**Livro:** começa na página **351** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## OTel Instrumentation · PDF p. 365

#### 🧪 Exercício: Rastreamento com spans

Escreva o gerenciador de contexto `span(nome)` que mede a duração, guarda o nome do *span* pai (pilha) e registra tudo numa lista. Instrumente um agente falso com 1 chamada de modelo e 2 ferramentas e imprima a árvore de *spans*.

In [ ]:
from contextlib import contextmanager
spans, pilha = [], []
@contextmanager
def span(nome):
    pai = pilha[-1] if pilha else None; pilha.append(nome); t = time.perf_counter()
    try: yield
    finally:
        pilha.pop(); spans.append({"nome": nome, "pai": pai, "ms": round((time.perf_counter() - t) * 1000)})
with span("agente"):
    with span("modelo.chamada"): time.sleep(0.2)
    with span("ferramenta.preco"): time.sleep(0.05)
    with span("ferramenta.pedido"): time.sleep(0.08)
for s in spans[::-1]: print(("   " if s["pai"] else "") + f"{s['nome']} ({s['ms']} ms)")

## Distribution Shifts · PDF p. 374

#### 🧪 Exercício: Mudança no tipo de pergunta (PSI)

Compare a distribuição de categorias de pedidos do mês passado com a deste mês usando o PSI (Σ (a − b)·ln(a/b)). Acima de 0,2 indica mudança importante.

In [ ]:
cats = ["preço", "pedido", "troca", "defeito"]
antes = np.array([0.40, 0.35, 0.15, 0.10]); agora = np.array([0.25, 0.30, 0.15, 0.30])
psi = float(np.sum((agora - antes) * np.log(agora / antes)))
print(round(psi, 3), "-> mudança importante" if psi > 0.2 else "-> estável")